# 12A 输入表征与模型双轴比较 + SIRIUS失败覆盖分析

**目的**：在冻结的Development结构簇五折上，分开回答两个问题：

1. **输入比较**：在统一线性预测头和共同化合物集合上，不同输入表征提供了多少P/B/T信息？
2. **完整管线比较**：各输入使用现有最优模型时，性能、覆盖率和失败模式如何？

本Notebook还会单独评价直接MS/MS模型在SIRIUS失败化合物上的表现，并构建预设的
`SIRIUS成功→SIRIUS；SIRIUS失败→直接MS/MS` 回退管线。

**边界条件**：

- 不读取ChemVista谱图或外部盲态标签；ChemVista继续锁定。
- SIRIUS缺失指纹不填0、不填均值，也不从Development总体中删除。
- `fingerprint_request_failed`仅表示超时、连接或其他非404异常；存在该状态时结果自动标记为`INTERIM`。
- `no_formula_candidates`及统一补算后仍稳定返回Fingerprint-not-found 404的`fingerprint_unavailable_after_retry`，计入方法覆盖局限。
- 结构指纹只作为需要已知结构的参考上限，不属于无结构部署方案。

In [ ]:
from __future__ import annotations

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import platform
import sys
import warnings

import numpy as np
import pandas as pd
from scipy import sparse
from scipy.stats import spearmanr
from sklearn import __version__ as sklearn_version
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    average_precision_score, balanced_accuracy_score, brier_score_loss,
    mean_absolute_error, mean_squared_error, r2_score, roc_auc_score,
)
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 220)
sns.set_theme(style='whitegrid', context='notebook')

SEED = 42
N_BOOTSTRAP = 1000
RUN_FIXED_HEAD_RETRAIN = True
FIXED_LOGISTIC_C = 1.0
FIXED_RIDGE_ALPHA = 10.0

cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents, Path(r'E:\谱图预测PBT_终版')]
ROOT = next((p for p in candidates if (p / 'outputs' / 'pbt_msms_v1').exists()), None)
assert ROOT is not None, '无法定位项目根目录，请将ROOT手动设置为项目目录。'

BASE = ROOT / 'outputs' / 'pbt_msms_v1'
SIMPLE_DIR = BASE / 'simple_spectrum_baselines_v1'
DREAMS_DIR = BASE / 'dreams_heads_v1'
TRANSFORMER_DIR = BASE / 'set_transformer_v1'
SIRIUS_DIR = BASE / 'sirius_heads_v1'
SIRIUS_RESULT_DIR = BASE / 'sirius_v1' / 'development_results_v1'
STRUCTURE_DIR = BASE / 'structure_baselines_v1'
OUTPUT_DIR = BASE / 'comparison_12a_v1'
FIGURE_DIR = OUTPUT_DIR / 'figures'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

TASKS = {
    'P_screen_not_ready_binary': {'label': 'P screening', 'column': 'label__P_screen_not_ready_binary', 'kind': 'classification'},
    'B_LOGBCF_REGRESSION': {'label': 'B logBCF', 'column': 'label__B_LOGBCF_REGRESSION', 'kind': 'regression'},
    'T_EC50_MG_L_REGRESSION': {'label': 'T EC50', 'column': 'label__T_EC50_MG_L_REGRESSION', 'kind': 'regression'},
    'T_LC50_MG_L_REGRESSION': {'label': 'T LC50', 'column': 'label__T_LC50_MG_L_REGRESSION', 'kind': 'regression'},
    'T_NOEC_MG_L_REGRESSION': {'label': 'T NOEC', 'column': 'label__T_NOEC_MG_L_REGRESSION', 'kind': 'regression'},
}

PATHS = {
    'simple_oof': SIMPLE_DIR / 'simple_spectrum_compound_level_oof.csv.gz',
    'simple_manifest': SIMPLE_DIR / 'simple_spectrum_run_manifest.json',
    'dreams_oof': DREAMS_DIR / 'dreams_head_compound_level_oof.csv.gz',
    'dreams_matrix': DREAMS_DIR / 'dreams_compound_embedding_routes.npz',
    'dreams_index': DREAMS_DIR / 'dreams_compound_embedding_index.csv',
    'dreams_manifest': DREAMS_DIR / 'dreams_head_run_manifest.json',
    'transformer_oof': TRANSFORMER_DIR / 'set_transformer_compound_level_oof.csv.gz',
    'transformer_manifest': TRANSFORMER_DIR / 'set_transformer_run_manifest.json',
    'sirius_oof': SIRIUS_DIR / 'sirius_head_compound_level_oof.csv.gz',
    'sirius_matrix': SIRIUS_DIR / 'sirius_compound_fingerprint_routes.npz',
    'sirius_index': SIRIUS_DIR / 'sirius_compound_fingerprint_index.csv',
    'sirius_feature_status': SIRIUS_DIR / 'sirius_feature_to_compound_status.csv.gz',
    'sirius_manifest': SIRIUS_DIR / 'sirius_head_run_manifest.json',
    'sirius_result_manifest': SIRIUS_RESULT_DIR / 'development_sirius_result_manifest.json',
    'structure_oof': STRUCTURE_DIR / 'structure_baseline_oof_predictions.csv.gz',
    'structure_matrix': STRUCTURE_DIR / 'structure_fingerprints.npz',
    'structure_index': STRUCTURE_DIR / 'structure_fingerprint_row_index.csv',
    'structure_manifest': STRUCTURE_DIR / 'structure_baseline_run_manifest.json',
}
missing = [str(p) for p in PATHS.values() if not p.exists()]
assert not missing, '缺少12A输入文件：\n' + '\n'.join(missing)

print('ROOT:', ROOT)
print('OUTPUT:', OUTPUT_DIR)
print('Python:', sys.executable)
print('scikit-learn:', sklearn_version)

## 1. 输入锁定、版本和SIRIUS技术失败门控

此处只读取Development阶段产物。若仍有`fingerprint_request_failed`，Notebook继续运行以便查看阶段性结果，
但运行清单不会将其标为正式冻结结果。

In [ ]:
def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()

manifests = {name: read_json(path) for name, path in PATHS.items() if name.endswith('manifest')}
assert manifests['simple_manifest'].get('external_test_unlocked') is False
assert manifests['dreams_manifest'].get('external_test_unlocked') is False
assert manifests['transformer_manifest'].get('external_test_unlocked') is False
assert manifests['structure_manifest'].get('external_test_unlocked') is False
assert 'ChemVista external evaluation remains locked' in manifests['sirius_manifest']['scientific_scope']

feature_status = pd.read_csv(PATHS['sirius_feature_status'])
feature_failure_counts = (
    feature_status.groupby(['mode', 'status'], dropna=False).size()
    .rename('n_features').reset_index()
)
technical_failures = int((feature_status['status'] == 'fingerprint_request_failed').sum())
no_formula = int((feature_status['status'] == 'no_formula_candidates').sum())
retry_unavailable = int((feature_status['status'] == 'fingerprint_unavailable_after_retry').sum())
coverage_limitations = no_formula + retry_unavailable
ANALYSIS_STATUS = 'FINAL_FROZEN' if technical_failures == 0 else 'INTERIM_TECHNICAL_FAILURES_PRESENT'
feature_failure_counts.to_csv(OUTPUT_DIR / '12A_sirius_feature_failure_counts.csv', index=False)

print(feature_failure_counts.to_string(index=False))
print('\ntechnical_failures:', technical_failures)
print('no_formula_candidates:', no_formula)
print('fingerprint_unavailable_after_retry:', retry_unavailable)
print('total_sirius_coverage_limitations:', coverage_limitations)
print('ANALYSIS_STATUS:', ANALYSIS_STATUS)
if technical_failures:
    print('\n警告：请在正式冻结12A前重试 fingerprint_request_failed，然后重新运行本Notebook。')

## 2. 统一指标和OOF数据格式

- P任务主指标：PR-AUC，越高越好。
- B/T回归主指标：MAE，越低越好。
- 所有统计以化合物为单位。
- 共同样本比较使用相同化合物的配对bootstrap。

In [ ]:
def classification_metrics(y_true, probability):
    y = np.asarray(y_true, dtype=int)
    p = np.asarray(probability, dtype=float)
    pred = (p >= 0.5).astype(int)
    return {
        'pr_auc': average_precision_score(y, p),
        'roc_auc': roc_auc_score(y, p) if len(np.unique(y)) == 2 else np.nan,
        'brier': brier_score_loss(y, p),
        'balanced_accuracy': balanced_accuracy_score(y, pred),
    }

def regression_metrics(y_true, prediction):
    y = np.asarray(y_true, dtype=float)
    p = np.asarray(prediction, dtype=float)
    rho = spearmanr(y, p).statistic if len(y) > 1 else np.nan
    return {
        'mae': mean_absolute_error(y, p),
        'rmse': mean_squared_error(y, p) ** 0.5,
        'r2': r2_score(y, p),
        'spearman': rho,
    }

def metric_dict(frame):
    kind = frame['kind'].iloc[0]
    if kind == 'classification':
        return classification_metrics(frame['y_true'], frame['y_score'])
    return regression_metrics(frame['y_true'], frame['y_score'])

def primary_metric(frame):
    metrics = metric_dict(frame)
    if frame['kind'].iloc[0] == 'classification':
        return metrics['pr_auc']
    return metrics['mae']

def bootstrap_metrics(frame, n_bootstrap=N_BOOTSTRAP, seed=SEED):
    base = metric_dict(frame)
    rng = np.random.default_rng(seed)
    values = {key: [] for key in base}
    n = len(frame)
    for _ in range(n_bootstrap):
        sample = frame.iloc[rng.integers(0, n, n)]
        try:
            result = metric_dict(sample)
        except ValueError:
            continue
        for key, value in result.items():
            if np.isfinite(value):
                values[key].append(value)
    row = {'n': n, **base}
    for key, samples in values.items():
        row[f'{key}_ci_low'] = np.quantile(samples, 0.025) if samples else np.nan
        row[f'{key}_ci_high'] = np.quantile(samples, 0.975) if samples else np.nan
    return row

def canonical_oof(frame, pipeline, *, representation_col='representation_route'):
    out = frame.copy()
    out['pipeline'] = pipeline
    out['representation_source'] = out[representation_col].astype(str)
    out['outer_fold'] = pd.to_numeric(out['outer_fold'], errors='raise').astype(int)
    out['y_score'] = np.where(
        out['kind'].eq('classification'),
        pd.to_numeric(out['y_probability'], errors='coerce'),
        pd.to_numeric(out['y_prediction'], errors='coerce'),
    )
    out['y_true'] = pd.to_numeric(out['y_true'], errors='coerce')
    keep = ['inchikey_2d', 'outer_fold', 'task', 'kind', 'pipeline',
            'representation_source', 'y_true', 'y_score']
    out = out[keep].dropna(subset=['inchikey_2d', 'y_true', 'y_score'])
    assert not out.duplicated(['inchikey_2d', 'task', 'pipeline']).any(), pipeline
    return out

## 3. 完整管线比较：读取各管线正式OOF预测

这里使用每条管线现有的预设主路线：简单谱图、DreaMS、Set Transformer、SIRIUS双模式可用均值，
以及Morgan结构指纹参考。母离子和元数据作为信息量下限。

In [ ]:
simple_raw = pd.read_csv(PATHS['simple_oof'], low_memory=False)
dreams_raw = pd.read_csv(PATHS['dreams_oof'], low_memory=False)
transformer_raw = pd.read_csv(PATHS['transformer_oof'], low_memory=False)
sirius_raw = pd.read_csv(PATHS['sirius_oof'], low_memory=False)
structure_raw = pd.read_csv(PATHS['structure_oof'], low_memory=False)

pipeline_frames = []
for route, label in [
    ('precursor_mz_only', 'Precursor m/z'),
    ('metadata_only', 'Metadata'),
    ('fragment_neutral_loss_metadata', 'Binned MS/MS'),
]:
    part = simple_raw[simple_raw['representation_route'].eq(route)]
    pipeline_frames.append(canonical_oof(part, label))

part = dreams_raw[
    dreams_raw['representation_route'].eq('all_spectra_compound_median')
    & dreams_raw['model'].eq('nested_selected_primary')
]
pipeline_frames.append(canonical_oof(part, 'DreaMS + selected head'))

part = transformer_raw[transformer_raw['representation_route'].eq('set_transformer_pretrained')]
pipeline_frames.append(canonical_oof(part, 'Set Transformer pretrained'))

part = sirius_raw[
    sirius_raw['representation_route'].eq('available_mode_mean')
    & sirius_raw['model'].eq('nested_selected_primary')
]
pipeline_frames.append(canonical_oof(part, 'SIRIUS probability fingerprint'))

task_model = np.where(structure_raw['kind'].eq('classification'), 'nested_selected', 'nested_selected')
part = structure_raw[
    structure_raw['representation'].eq('Morgan_2048')
    & structure_raw['model'].eq('nested_selected')
]
pipeline_frames.append(canonical_oof(part, 'Morgan structure reference', representation_col='representation'))

pipeline_oof = pd.concat(pipeline_frames, ignore_index=True)
expected_pipelines = [
    'Precursor m/z', 'Metadata', 'Binned MS/MS', 'DreaMS + selected head',
    'Set Transformer pretrained', 'SIRIUS probability fingerprint', 'Morgan structure reference',
]
assert set(pipeline_oof['pipeline']) == set(expected_pipelines)
pipeline_oof.to_csv(OUTPUT_DIR / '12A_pipeline_oof_long.csv.gz', index=False, compression='gzip')
print(pipeline_oof.groupby(['pipeline', 'task']).size().unstack(fill_value=0))

## 4. 全Development覆盖率和成功条件下性能

分母是全部具有对应标签的Development化合物。性能只在该管线实际产生OOF预测的样本上计算，
同时强制报告覆盖率，避免把失败样本静默删除。

In [ ]:
sirius_index = pd.read_csv(PATHS['sirius_index'], low_memory=False)
master_columns = ['inchikey_2d', 'cv_group', 'cv_fold'] + [x['column'] for x in TASKS.values()]
master = sirius_index[master_columns].copy()
assert master['inchikey_2d'].is_unique and len(master) == 1238

labelled_denominators = {
    task: int(pd.to_numeric(master[info['column']], errors='coerce').notna().sum())
    for task, info in TASKS.items()
}

native_rows = []
for (pipeline, task), group in tqdm(pipeline_oof.groupby(['pipeline', 'task']), desc='原生覆盖指标'):
    result = bootstrap_metrics(group, seed=SEED + sum(map(ord, pipeline + task)))
    denominator = labelled_denominators[task]
    native_rows.append({
        'analysis': 'native_coverage', 'pipeline': pipeline, 'task': task,
        'kind': group['kind'].iloc[0], 'n_labelled_development': denominator,
        'n_predictable': len(group), 'coverage': len(group) / denominator, **result,
    })
native_metrics = pd.DataFrame(native_rows)
native_metrics.to_csv(OUTPUT_DIR / '12A_native_coverage_metrics.csv', index=False)
display(native_metrics[['pipeline', 'task', 'n_predictable', 'n_labelled_development', 'coverage', 'pr_auc', 'mae', 'spearman']])

## 5. SIRIUS化合物级覆盖与失败样本

`has_available_mode_mean=False`表示正负模式均未获得可用概率指纹。模式没有输入谱图和模式计算失败分开记录。

In [ ]:
sirius_availability = sirius_index[[
    'inchikey_2d', 'cv_fold',
    'n_positive_features_total', 'n_positive_features_success', 'has_positive_only',
    'n_negative_features_total', 'n_negative_features_success', 'has_negative_only',
    'has_available_mode_mean', 'n_available_modes',
]].copy()
for col in ['has_positive_only', 'has_negative_only', 'has_available_mode_mean']:
    sirius_availability[col] = sirius_availability[col].astype(str).str.lower().eq('true')
sirius_availability['positive_state'] = np.select(
    [sirius_availability['has_positive_only'], sirius_availability['n_positive_features_total'].eq(0)],
    ['success', 'no_positive_input'], default='all_positive_features_failed')
sirius_availability['negative_state'] = np.select(
    [sirius_availability['has_negative_only'], sirius_availability['n_negative_features_total'].eq(0)],
    ['success', 'no_negative_input'], default='all_negative_features_failed')
sirius_availability['compound_route'] = np.select(
    [
        sirius_availability['has_positive_only'] & sirius_availability['has_negative_only'],
        sirius_availability['has_positive_only'],
        sirius_availability['has_negative_only'],
    ],
    ['both_modes', 'positive_only', 'negative_only'], default='no_sirius_fingerprint')
sirius_availability.to_csv(OUTPUT_DIR / '12A_sirius_compound_availability.csv', index=False)
compound_counts = sirius_availability['compound_route'].value_counts().rename_axis('compound_route').reset_index(name='n_compounds')
compound_counts['fraction'] = compound_counts['n_compounds'] / len(sirius_availability)
compound_counts.to_csv(OUTPUT_DIR / '12A_sirius_compound_route_counts.csv', index=False)
display(compound_counts)

## 6. 共同样本上的公平管线比较

对每个任务取所有管线都有预测的化合物交集。此结果隔离了样本组成差异，但它是“条件于SIRIUS成功”的比较，
必须与上一节覆盖率结果一起解释。

In [ ]:
common_parts = []
common_counts = []
for task, task_frame in pipeline_oof.groupby('task'):
    counts = task_frame.groupby('inchikey_2d')['pipeline'].nunique()
    common_keys = counts[counts.eq(len(expected_pipelines))].index
    part = task_frame[task_frame['inchikey_2d'].isin(common_keys)].copy()
    per_pipeline_n = part.groupby('pipeline')['inchikey_2d'].nunique()
    assert per_pipeline_n.nunique() == 1
    truth_check = part.groupby('inchikey_2d')['y_true'].nunique()
    fold_check = part.groupby('inchikey_2d')['outer_fold'].nunique()
    assert truth_check.max() == 1 and fold_check.max() == 1
    common_parts.append(part)
    common_counts.append({'task': task, 'n_common_compounds': len(common_keys)})

common_oof = pd.concat(common_parts, ignore_index=True)
common_metrics_rows = []
for (pipeline, task), group in tqdm(common_oof.groupby(['pipeline', 'task']), desc='共同样本指标'):
    common_metrics_rows.append({
        'analysis': 'all_pipeline_common_subset', 'pipeline': pipeline, 'task': task,
        'kind': group['kind'].iloc[0], **bootstrap_metrics(group, seed=SEED + sum(map(ord, pipeline + task))),
    })
common_metrics = pd.DataFrame(common_metrics_rows)
pd.DataFrame(common_counts).to_csv(OUTPUT_DIR / '12A_common_subset_counts.csv', index=False)
common_metrics.to_csv(OUTPUT_DIR / '12A_common_subset_metrics.csv', index=False)
common_oof.to_csv(OUTPUT_DIR / '12A_common_subset_oof_long.csv.gz', index=False, compression='gzip')
display(pd.DataFrame(common_counts))
display(common_metrics[['pipeline', 'task', 'n', 'pr_auc', 'mae', 'spearman']])

## 7. 以SIRIUS为参照的配对bootstrap

分类差值为`比较管线PR-AUC − SIRIUS PR-AUC`；回归差值为`SIRIUS MAE − 比较管线MAE`。
因而两类任务均为正值表示比较管线更好。

In [ ]:
def paired_bootstrap_against_sirius(task_frame, comparator, n_bootstrap=N_BOOTSTRAP):
    ref_name = 'SIRIUS probability fingerprint'
    ref = task_frame[task_frame['pipeline'].eq(ref_name)][['inchikey_2d', 'y_true', 'y_score', 'kind']]
    cmp = task_frame[task_frame['pipeline'].eq(comparator)][['inchikey_2d', 'y_true', 'y_score']]
    paired = ref.merge(cmp, on=['inchikey_2d', 'y_true'], suffixes=('_ref', '_cmp'), validate='one_to_one')
    kind = paired['kind'].iloc[0]

    def delta(frame):
        if kind == 'classification':
            return (
                average_precision_score(frame['y_true'], frame['y_score_cmp'])
                - average_precision_score(frame['y_true'], frame['y_score_ref'])
            )
        return (
            mean_absolute_error(frame['y_true'], frame['y_score_ref'])
            - mean_absolute_error(frame['y_true'], frame['y_score_cmp'])
        )

    observed = delta(paired)
    rng = np.random.default_rng(SEED + sum(map(ord, comparator + str(paired['y_true'].sum()))))
    samples = []
    for _ in range(n_bootstrap):
        sample = paired.iloc[rng.integers(0, len(paired), len(paired))]
        try:
            value = delta(sample)
        except ValueError:
            continue
        if np.isfinite(value):
            samples.append(value)
    samples = np.asarray(samples)
    p_two_sided = min(1.0, 2 * min(np.mean(samples <= 0), np.mean(samples >= 0)))
    return {
        'comparator': comparator, 'reference': ref_name, 'kind': kind,
        'n_common': len(paired), 'delta_positive_is_comparator_better': observed,
        'ci_low': np.quantile(samples, 0.025), 'ci_high': np.quantile(samples, 0.975),
        'bootstrap_p_two_sided': p_two_sided,
    }

paired_rows = []
for task, frame in common_oof.groupby('task'):
    for comparator in expected_pipelines:
        if comparator == 'SIRIUS probability fingerprint':
            continue
        paired_rows.append({'task': task, **paired_bootstrap_against_sirius(frame, comparator)})
paired_bootstrap = pd.DataFrame(paired_rows)
paired_bootstrap.to_csv(OUTPUT_DIR / '12A_paired_bootstrap_vs_sirius.csv', index=False)
display(paired_bootstrap)

## 8. 固定线性头的输入表征比较

为避免“输入变化”和“模型变化”混在一起，本节使用固定模型：分类`LogisticRegression(C=1)`，
回归`Ridge(alpha=10)`，严格沿用冻结外层结构簇五折。DreaMS在每个训练折内标准化；
SIRIUS概率指纹和Morgan位指纹不做缺失填补。

母离子、元数据和分箱MS/MS使用05中相同固定头产生的现有OOF；DreaMS、SIRIUS和Morgan在本节重新训练固定头。

In [ ]:
def fixed_linear_oof(matrix, index, input_name, *, standardize=False):
    rows = []
    matrix = np.asarray(matrix) if not sparse.issparse(matrix) else matrix.tocsr()
    finite_rows = np.isfinite(matrix).all(axis=1) if not sparse.issparse(matrix) else np.ones(matrix.shape[0], dtype=bool)
    if hasattr(finite_rows, 'A1'):
        finite_rows = finite_rows.A1
    for task, info in TASKS.items():
        labels = pd.to_numeric(index[info['column']], errors='coerce').to_numpy(float)
        valid = np.isfinite(labels) & finite_rows
        for fold in range(5):
            train = valid & index['cv_fold'].ne(fold).to_numpy()
            test = valid & index['cv_fold'].eq(fold).to_numpy()
            X_train, X_test = matrix[train], matrix[test]
            if standardize:
                scaler = StandardScaler(with_mean=not sparse.issparse(X_train)).fit(X_train)
                X_train, X_test = scaler.transform(X_train), scaler.transform(X_test)
            if info['kind'] == 'classification':
                model = LogisticRegression(
                    C=FIXED_LOGISTIC_C, solver='liblinear', max_iter=4000,
                    random_state=SEED + fold,
                )
                model.fit(X_train, labels[train].astype(int))
                score = model.predict_proba(X_test)[:, 1]
            else:
                model = Ridge(alpha=FIXED_RIDGE_ALPHA, solver='lsqr', tol=1e-4, max_iter=3000)
                model.fit(X_train, labels[train])
                score = model.predict(X_test)
            test_rows = index.loc[test, ['inchikey_2d', 'cv_fold']].copy()
            test_rows['outer_fold'] = fold
            test_rows['task'] = task
            test_rows['kind'] = info['kind']
            test_rows['pipeline'] = input_name
            test_rows['representation_source'] = input_name
            test_rows['y_true'] = labels[test]
            test_rows['y_score'] = score
            rows.append(test_rows.drop(columns='cv_fold'))
    out = pd.concat(rows, ignore_index=True)
    assert not out.duplicated(['inchikey_2d', 'task', 'pipeline']).any()
    return out

fixed_existing = pipeline_oof[pipeline_oof['pipeline'].isin(['Precursor m/z', 'Metadata', 'Binned MS/MS'])].copy()
fixed_frames = [fixed_existing]

if RUN_FIXED_HEAD_RETRAIN:
    label_master = master.set_index('inchikey_2d')

    dreams_npz = np.load(PATHS['dreams_matrix'], allow_pickle=False)
    dreams_index = pd.read_csv(PATHS['dreams_index'], low_memory=False)
    for task, info in TASKS.items():
        dreams_index[info['column']] = dreams_index['inchikey_2d'].map(label_master[info['column']])
    fixed_frames.append(fixed_linear_oof(
        dreams_npz['all_spectra'], dreams_index, 'DreaMS embedding | fixed linear', standardize=True,
    ))

    sirius_npz = np.load(PATHS['sirius_matrix'], allow_pickle=False)
    fixed_frames.append(fixed_linear_oof(
        sirius_npz['available_mode_mean'], master, 'SIRIUS fingerprint | fixed linear', standardize=False,
    ))

    structure_npz = np.load(PATHS['structure_matrix'], allow_pickle=False)
    structure_index = pd.read_csv(PATHS['structure_index'], low_memory=False)
    for task, info in TASKS.items():
        structure_index[info['column']] = structure_index['inchikey_2d'].map(label_master[info['column']])
    fixed_frames.append(fixed_linear_oof(
        structure_npz['morgan'], structure_index, 'Morgan structure | fixed linear', standardize=False,
    ))

fixed_oof = pd.concat(fixed_frames, ignore_index=True)
fixed_inputs = fixed_oof['pipeline'].drop_duplicates().tolist()
fixed_common_parts = []
for task, frame in fixed_oof.groupby('task'):
    counts = frame.groupby('inchikey_2d')['pipeline'].nunique()
    keys = counts[counts.eq(len(fixed_inputs))].index
    fixed_common_parts.append(frame[frame['inchikey_2d'].isin(keys)])
fixed_common_oof = pd.concat(fixed_common_parts, ignore_index=True)

fixed_metric_rows = []
for (pipeline, task), group in tqdm(fixed_common_oof.groupby(['pipeline', 'task']), desc='固定头共同样本指标'):
    fixed_metric_rows.append({
        'analysis': 'fixed_linear_common_subset', 'pipeline': pipeline, 'task': task,
        'kind': group['kind'].iloc[0], **bootstrap_metrics(group, seed=SEED + sum(map(ord, pipeline + task))),
    })
fixed_metrics = pd.DataFrame(fixed_metric_rows)
fixed_oof.to_csv(OUTPUT_DIR / '12A_fixed_linear_oof_long.csv.gz', index=False, compression='gzip')
fixed_metrics.to_csv(OUTPUT_DIR / '12A_fixed_linear_common_subset_metrics.csv', index=False)
display(fixed_metrics[['pipeline', 'task', 'n', 'pr_auc', 'mae', 'spearman']])

## 9. 同一输入内部的模型比较

模型比较不能跨输入直接归因。本节分别整理DreaMS模型族、Morgan结构模型族和Set Transformer消融路线。
SIRIUS当前只保存了嵌套选择后的主OOF，因此其模型族扩展留到需要时单独运行。

In [ ]:
model_frames = []

dreams_models = dreams_raw[
    dreams_raw['representation_route'].eq('all_spectra_compound_median')
    & dreams_raw['model'].isin(['linear', 'mlp', 'random_forest', 'catboost', 'svm_rbf'])
].copy()
for model, part in dreams_models.groupby('model'):
    model_frames.append(canonical_oof(part, f'DreaMS | {model}'))

transformer_models = transformer_raw.copy()
for route, part in transformer_models.groupby('representation_route'):
    model_frames.append(canonical_oof(part, f'SetTransformer | {route}'))

structure_models = structure_raw[structure_raw['representation'].eq('Morgan_2048')].copy()
allowed_structure = ['logistic', 'ridge', 'random_forest', 'catboost', 'svc_rbf', 'svr_rbf']
for model, part in structure_models[structure_models['model'].isin(allowed_structure)].groupby('model'):
    model_frames.append(canonical_oof(part, f'Morgan | {model}', representation_col='representation'))

model_oof = pd.concat(model_frames, ignore_index=True)
model_rows = []
for (pipeline, task), group in model_oof.groupby(['pipeline', 'task']):
    model_rows.append({'pipeline_model': pipeline, 'task': task, 'kind': group['kind'].iloc[0],
                       'n': len(group), **metric_dict(group)})
model_metrics = pd.DataFrame(model_rows)
model_metrics.to_csv(OUTPUT_DIR / '12A_within_input_model_family_metrics.csv', index=False)
display(model_metrics)

## 10. 直接MS/MS模型在SIRIUS失败样本上的表现及回退管线

这是12A最重要的应用分析之一：SIRIUS失败样本不会从评价中消失，而是由直接谱图模型继续预测。
回退规则在查看外部数据之前固定，并且所有组成预测均为OOF预测。

In [ ]:
availability_flag = sirius_availability.set_index('inchikey_2d')['has_available_mode_mean']
direct_names = ['Binned MS/MS', 'DreaMS + selected head', 'Set Transformer pretrained']
direct_rows = []
for (pipeline, task), group in pipeline_oof[pipeline_oof['pipeline'].isin(direct_names)].groupby(['pipeline', 'task']):
    group = group.copy()
    group['sirius_available'] = group['inchikey_2d'].map(availability_flag).fillna(False)
    for available, subgroup in group.groupby('sirius_available'):
        if len(subgroup) < 10:
            continue
        direct_rows.append({
            'pipeline': pipeline, 'task': task, 'kind': subgroup['kind'].iloc[0],
            'sirius_subset': 'success' if available else 'failure',
            'n': len(subgroup), **metric_dict(subgroup),
        })
direct_failure_metrics = pd.DataFrame(direct_rows)
direct_failure_metrics.to_csv(OUTPUT_DIR / '12A_direct_models_on_sirius_success_failure.csv', index=False)

def make_fallback(direct_pipeline):
    sirius = pipeline_oof[pipeline_oof['pipeline'].eq('SIRIUS probability fingerprint')]
    direct = pipeline_oof[pipeline_oof['pipeline'].eq(direct_pipeline)]
    rows = []
    for task in TASKS:
        s = sirius[sirius['task'].eq(task)].set_index('inchikey_2d')
        d = direct[direct['task'].eq(task)].set_index('inchikey_2d')
        keys = s.index.union(d.index)
        for key in keys:
            source = 'SIRIUS' if key in s.index else direct_pipeline
            row = s.loc[key] if key in s.index else d.loc[key]
            rows.append({
                'inchikey_2d': key, 'outer_fold': int(row['outer_fold']), 'task': task,
                'kind': row['kind'], 'pipeline': f'SIRIUS -> {direct_pipeline} fallback',
                'representation_source': source, 'y_true': float(row['y_true']),
                'y_score': float(row['y_score']),
            })
    return pd.DataFrame(rows)

fallback_oof = pd.concat([
    make_fallback('DreaMS + selected head'),
    make_fallback('Set Transformer pretrained'),
], ignore_index=True)
fallback_rows = []
for (pipeline, task), group in fallback_oof.groupby(['pipeline', 'task']):
    denominator = labelled_denominators[task]
    fallback_rows.append({
        'pipeline': pipeline, 'task': task, 'kind': group['kind'].iloc[0],
        'n_labelled_development': denominator, 'n_predictable': len(group),
        'coverage': len(group) / denominator, **bootstrap_metrics(group, seed=SEED + sum(map(ord, pipeline + task))),
    })
fallback_metrics = pd.DataFrame(fallback_rows)
fallback_oof.to_csv(OUTPUT_DIR / '12A_fallback_pipeline_oof.csv.gz', index=False, compression='gzip')
fallback_metrics.to_csv(OUTPUT_DIR / '12A_fallback_pipeline_metrics.csv', index=False)
display(direct_failure_metrics)
display(fallback_metrics[['pipeline', 'task', 'coverage', 'pr_auc', 'mae', 'spearman']])

## 11. 图表草稿

输出三张12A工作图：共同样本性能、原生覆盖率、SIRIUS失败与回退管线。正式论文作图时可在此基础上统一配色和排版。

In [ ]:
order = expected_pipelines
colors = sns.color_palette('colorblind', n_colors=len(order))
color_map = dict(zip(order, colors))

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5), constrained_layout=True)
p_metrics = common_metrics[common_metrics['task'].eq('P_screen_not_ready_binary')].copy()
p_metrics['pipeline'] = pd.Categorical(p_metrics['pipeline'], order, ordered=True)
p_metrics = p_metrics.sort_values('pipeline')
axes[0].barh(p_metrics['pipeline'], p_metrics['pr_auc'], color=[color_map[str(x)] for x in p_metrics['pipeline']])
axes[0].set_xlabel('PR-AUC (higher is better)')
axes[0].set_title('A  P classification | common compounds')
axes[0].set_xlim(0, 1)

reg = common_metrics[common_metrics['kind'].eq('regression')].copy()
heat = reg.pivot(index='pipeline', columns='task', values='mae').reindex(order)
sns.heatmap(heat, annot=True, fmt='.3f', cmap='mako_r', ax=axes[1], cbar_kws={'label': 'MAE (lower is better)'})
axes[1].set_title('B  B/T regression | common compounds')
axes[1].set_xlabel('Endpoint')
axes[1].set_ylabel('')
fig.savefig(FIGURE_DIR / 'Fig12A_common_subset_performance.png', dpi=300, bbox_inches='tight')
fig.savefig(FIGURE_DIR / 'Fig12A_common_subset_performance.pdf', bbox_inches='tight')
plt.show()

coverage = native_metrics.pivot(index='pipeline', columns='task', values='coverage').reindex(order)
fig, ax = plt.subplots(figsize=(12, 6), constrained_layout=True)
sns.heatmap(coverage, annot=True, fmt='.1%', cmap='crest', vmin=0, vmax=1, ax=ax, cbar_kws={'label': 'Coverage'})
ax.set_title('Native pipeline coverage on all labelled Development compounds')
ax.set_xlabel('Endpoint')
ax.set_ylabel('')
fig.savefig(FIGURE_DIR / 'Fig12A_native_coverage.png', dpi=300, bbox_inches='tight')
fig.savefig(FIGURE_DIR / 'Fig12A_native_coverage.pdf', bbox_inches='tight')
plt.show()

fallback_plot = fallback_metrics.copy()
fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
sns.barplot(data=compound_counts, x='compound_route', y='n_compounds', ax=axes[0], color='#4472C4')
axes[0].set_title('A  SIRIUS compound-level availability')
axes[0].tick_params(axis='x', rotation=20)
axes[0].set_xlabel('')
sns.heatmap(
    fallback_plot.pivot(index='pipeline', columns='task', values='coverage'),
    annot=True, fmt='.1%', cmap='crest', vmin=0, vmax=1, ax=axes[1],
    cbar_kws={'label': 'Coverage'},
)
axes[1].set_title('B  Coverage after pre-specified fallback')
axes[1].set_xlabel('Endpoint')
axes[1].set_ylabel('')
fig.savefig(FIGURE_DIR / 'Fig12A_sirius_failure_and_fallback.png', dpi=300, bbox_inches='tight')
fig.savefig(FIGURE_DIR / 'Fig12A_sirius_failure_and_fallback.pdf', bbox_inches='tight')
plt.show()

## 12. 最终QC、运行清单和判读顺序

**判读顺序**：先看SIRIUS技术失败是否清零，再看覆盖率，然后看共同样本性能和配对差值，最后评价回退管线。
在`ANALYSIS_STATUS`仍为`INTERIM_TECHNICAL_FAILURES_PRESENT`时，不要把表格数字写成最终论文结果。

In [ ]:
qc_rows = []
def add_qc(check_id, description, passed, detail=''):
    qc_rows.append({'check_id': check_id, 'description': description,
                    'status': 'PASS' if passed else 'FAIL', 'detail': detail})

add_qc('12A001', 'Development主表ID唯一', master['inchikey_2d'].is_unique, f'n={len(master)}')
add_qc('12A002', '所有管线预测键唯一', not pipeline_oof.duplicated(['pipeline', 'task', 'inchikey_2d']).any())
add_qc('12A003', '所有外折均为0-4', set(pipeline_oof['outer_fold'].unique()) == set(range(5)))
add_qc('12A004', '所有预测分数有限', np.isfinite(pipeline_oof['y_score']).all())
add_qc('12A005', '覆盖率位于0-1', native_metrics['coverage'].between(0, 1).all())
add_qc('12A006', '共同样本中每任务各管线n一致',
       common_metrics.groupby('task')['n'].nunique().eq(1).all())
add_qc('12A007', 'SIRIUS失败指纹未进入SIRIUS OOF',
       set(sirius_raw['inchikey_2d']).isdisjoint(
           set(sirius_availability.loc[~sirius_availability['has_available_mode_mean'], 'inchikey_2d'])
       ))
add_qc('12A008', '回退管线预测键唯一',
       not fallback_oof.duplicated(['pipeline', 'task', 'inchikey_2d']).any())
add_qc('12A009', '未读取ChemVista谱图路径',
       not any('chemvista' in str(path).lower() for path in PATHS.values()))
qc_rows.append({
    'check_id': '12A010', 'description': '正式状态要求技术失败清零',
    'status': 'PASS' if technical_failures == 0 else 'WARN',
    'detail': (f'technical_failures={technical_failures}; '               f'fingerprint_unavailable_after_retry={retry_unavailable}; '               f'no_formula_candidates={no_formula}'),
})

qc = pd.DataFrame(qc_rows)
qc.to_csv(OUTPUT_DIR / '12A_qc_checks.csv', index=False)
display(qc)

formal_outputs = [
    OUTPUT_DIR / '12A_sirius_feature_failure_counts.csv',
    OUTPUT_DIR / '12A_sirius_compound_availability.csv',
    OUTPUT_DIR / '12A_sirius_compound_route_counts.csv',
    OUTPUT_DIR / '12A_pipeline_oof_long.csv.gz',
    OUTPUT_DIR / '12A_native_coverage_metrics.csv',
    OUTPUT_DIR / '12A_common_subset_counts.csv',
    OUTPUT_DIR / '12A_common_subset_metrics.csv',
    OUTPUT_DIR / '12A_common_subset_oof_long.csv.gz',
    OUTPUT_DIR / '12A_paired_bootstrap_vs_sirius.csv',
    OUTPUT_DIR / '12A_fixed_linear_oof_long.csv.gz',
    OUTPUT_DIR / '12A_fixed_linear_common_subset_metrics.csv',
    OUTPUT_DIR / '12A_within_input_model_family_metrics.csv',
    OUTPUT_DIR / '12A_direct_models_on_sirius_success_failure.csv',
    OUTPUT_DIR / '12A_fallback_pipeline_oof.csv.gz',
    OUTPUT_DIR / '12A_fallback_pipeline_metrics.csv',
    OUTPUT_DIR / '12A_qc_checks.csv',
    FIGURE_DIR / 'Fig12A_common_subset_performance.png',
    FIGURE_DIR / 'Fig12A_common_subset_performance.pdf',
    FIGURE_DIR / 'Fig12A_native_coverage.png',
    FIGURE_DIR / 'Fig12A_native_coverage.pdf',
    FIGURE_DIR / 'Fig12A_sirius_failure_and_fallback.png',
    FIGURE_DIR / 'Fig12A_sirius_failure_and_fallback.pdf',
]
missing_outputs = [str(path) for path in formal_outputs if not path.exists()]
assert not missing_outputs, '缺少正式输出：\n' + '\n'.join(missing_outputs)

run_manifest = {
    'run_version': 'PBT_MSMS_COMPARISON_12A_V1',
    'generated_at_utc': datetime.now(timezone.utc).isoformat(),
    'analysis_status': ANALYSIS_STATUS,
    'scientific_scope': 'Development public spectra only; ChemVista external evaluation locked',
    'seed': SEED,
    'bootstrap_resamples': N_BOOTSTRAP,
    'comparison_layers': {
        'fixed_head_common_subset': 'Fixed logistic C=1 / ridge alpha=10; frozen 5-fold structure clusters',
        'optimized_pipeline_common_subset': 'Existing pre-specified main OOF routes on identical compounds',
        'native_coverage': 'Available predictions divided by all labelled Development compounds',
        'fallback': 'Use SIRIUS OOF when available; otherwise direct-MS/MS OOF',
    },
    'sirius_failure_policy': {
        'fingerprint_request_failed': 'technical failure; timeout, connection, server, or other non-404 exception',
        'fingerprint_unavailable_after_retry': 'coverage limitation after uniform frozen-setting retry; no zero/mean imputation',
        'no_formula_candidates': 'coverage limitation; no zero/mean imputation',
        'technical_failure_count': technical_failures,
        'fingerprint_unavailable_after_retry_count': retry_unavailable,
        'no_formula_candidate_count': no_formula,
        'total_coverage_limitation_count': coverage_limitations,
    },
    'software': {
        'python': sys.version, 'platform': platform.platform(),
        'numpy': np.__version__, 'pandas': pd.__version__,
        'scikit_learn': sklearn_version,
    },
    'input_hashes': {name: sha256_file(path) for name, path in PATHS.items()},
    'output_hashes': {path.name: sha256_file(path) for path in formal_outputs},
}
manifest_path = OUTPUT_DIR / '12A_run_manifest.json'
manifest_path.write_text(json.dumps(run_manifest, ensure_ascii=False, indent=2), encoding='utf-8')
print('\n12A完成。状态:', ANALYSIS_STATUS)
print('输出目录:', OUTPUT_DIR)
if technical_failures:
    print('正式冻结前：仍需处理 fingerprint_request_failed。')
else:
    print('SIRIUS技术失败已清零；补算后不可用指纹按覆盖限制保留。')